# Scene3D

## I. Getting Started Guide

### 1. Scene3D Model Introduction

Scene3D is a monocular depth estimation network within the VisionPilot suite designed to infer per-pixel relative scene depth from a single front-view image. It follows an Encoder-Context-Decoder pattern, combining pretrained semantic features with a dedicated depth-context module to recover geometry-aware structure.

The architecture components include:

- **Pretrained Backbone (`PreTrainedBackbone`)**: wraps the frozen `EfficientNet-B0` encoder from `SceneSegNetwork` to extract hierarchical multi-scale features from the input image.

- **Depth Context (`DepthContext`)**: consumes the deepest feature map, performs global spatial pooling, passes it through MLP layers, reshapes the compact representation into a 2D context map, and expands it with convolutional layers.

- **Residual Attention Fusion**: injects context into deep features using $Context \times Features + Features$, helping the model encode global scene layout while preserving local feature detail.

- **Scene3D Neck (`Scene3DNeck`)**: progressive transposed-convolution decoder with skip connections from encoder stages (`features[3]`, `features[2]`, `features[1]`) to recover spatial resolution and fine structural cues.

- **Scene3D Head (`Scene3DHead`)**: performs the final upsampling with an additional skip connection from `features[0]`, followed by convolutional prediction layers that produce a single-channel dense depth output (`320px × 640px`).

As a scene geometry model, Scene3D processes raw perspective frames and predicts dense depth for each pixel in real time. This allows downstream driving stacks to estimate free space and object distance cues without requiring LiDAR, while remaining lightweight enough for embedded deployment workflows.

![](Media/Scene3D_GIF_2.gif)

### 2. Environment Setup

First, if you have not already cloned the repository locally, clone `scene_3d` from the Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()
print(base_dir)

# Clone scene_3d repo if running outside the repository
if not os.path.exists("./Models"):
    if not os.path.exists("./scene_3d"):
        !git clone https://github.com/autowarefoundation/scene_3d.git
    %cd scene_3d
    base_dir = os.getcwd()

The Scene3D project relies on several key libraries, including **PyTorch** for model execution and **OpenCV** for image processing.

We install the required dependencies directly from `requirements.txt` at the repository root.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r ./requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files using the links below. Subsequent tutorial sections assume they are saved inside `./weights/`.

- [Link to Download PyTorch Model Weights (*.pth)](https://drive.google.com/file/d/1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B/view?usp=sharing)
- [Link to Download Traced PyTorch Model (*.pt)](https://drive.google.com/file/d/1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0/view?usp=drive_link)
- [Link to Download ONNX FP32 Weights (*.onnx)](https://drive.google.com/file/d/19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6/view?usp=drive_link)

You can also run the cell below, which uses `gdown` to download the model weights into `./weights/`.

In [ ]:
%pip install gdown

import os
import gdown

base_dir = os.getcwd()
model_dir = "./weights"
os.makedirs(model_dir, exist_ok=True)

metadata = {
    "pth": {
        "url": "https://drive.google.com/uc?id=1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B",
        "filepath": os.path.join(base_dir, model_dir, "scene3d.pth"),
    },
    "pt": {
        "url": "https://drive.google.com/uc?id=1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0",
        "filepath": os.path.join(base_dir, model_dir, "scene3d.pt"),
    },
    "onnx": {
        "url": "https://drive.google.com/uc?id=19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6",
        "filepath": os.path.join(base_dir, model_dir, "scene3d.onnx"),
    },
}

for weight_type, info in metadata.items():
    url = info["url"]
    filepath = info["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading Scene3D {weight_type} weights...")
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"Scene3D {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the **Environment Setup** and **Download Models** steps above.

### 1. Image Inference

You can run inference directly via `Scene3DNetworkInfer` (PyTorch `.pth` / `.pt`) or `Scene3DONNXInfer` (ONNX Runtime `.onnx`), or batch-process images using `Models/visualizations/image_visualization.py`, which overlays colorized viridis depth predictions onto the original frames.

#### a. Run Batch Image Inference

Key script parameters for `Models/visualizations/image_visualization.py`:
- `-p / --model_checkpoint_path`: path to your `.pth`, `.pt`, or `.onnx` weights file.
- `-i / --input_image_dirpath`: image file or directory containing `.png`, `.jpg`, or `.jpeg` files to process.
- `-o / --output_image_dirpath`: destination directory where visualization files are saved as `[image_id]_result.png`.

In [ ]:
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(base_dir, "Media")
OUTPUT_DIR = os.path.join(base_dir, "results", "images")

!python3 Models/visualizations/image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running inference, we can use matplotlib and PIL to visualize the generated depth overlays directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith("_result.png")
])

if result_files:
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))
    for i, ax in enumerate(axes):
        if i < len(result_files):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we apply the Scene3D model to video streams using `Models/visualizations/video_visualization.py`, which processes video files frame-by-frame (supporting `.pth`, `.pt`, and `.onnx` models) and compiles the colorized viridis depth overlays into a new `.avi` video.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained Scene3D `.pth`, `.pt`, or `.onnx` weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_filepath`: destination output path for the visualized video (`.avi` appended if omitted).
- `-v / --vis`: optional flag to enable a pop-up window showing frames as they are processed.

Technical details:

- The script resizes each frame to $(640, 320)$ before inference, then resizes visualization outputs to $(1280, 720)$ for the final MJPG `.avi` video.
- It applies a viridis colormap and alpha blending with $\alpha = 0.97$ to produce a depth-overlay visualization.
- Video capture and writer resources are released automatically upon completion.

In [ ]:
INPUT_VIDEO_PATH = os.path.join(base_dir, "Media", "highway_normal.mp4")
OUTPUT_VIDEO_PATH = os.path.join(base_dir, "results", "videos", "highway_normal_output.avi")

!python3 Models/visualizations/video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_VIDEO_PATH)} \
    -o {os.path.abspath(OUTPUT_VIDEO_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained SceneSeg backbone or resume from Scene3D checkpoint

Model loading behavior is controlled in `Models/training/train_scene_3d.py` via `Scene3DTrainer(...)`:

- If `--load_from_save` (`-l`) is **not** provided, the script creates `Scene3DTrainer(pretrained_checkpoint_path=...)`. In this mode, the upstream `SceneSeg` backbone is initialized from a pre-trained checkpoint (`-m`), while Scene3D downstream layers are trained from random initialization.
- If `--load_from_save` (`-l`) **is** provided, the script creates `Scene3DTrainer(checkpoint_path=..., is_pretrained=True)`. In this mode, the full Scene3D network is restored from `-c` and training continues from that saved state.

Practical recommendation:
- Use `-m` when starting a fresh Scene3D run with only upstream SceneSeg pretraining available.
- Use `-l -c` when continuing or fine-tuning an existing Scene3D training run.

#### b. Prepare the training datasets

Scene3D data preparation supports two label-generation styles:

1. **Relative-depth pseudo labels** (generated with DepthAnythingV2-Large — see `Models/data_parsing/README.md`)
2. **Metric-depth labels from LiDAR/stereo datasets**

The scripts in `Models/data_parsing/` support metric-depth dataset creation from:
- Argoverse (`Argoverse/process_argoverse.py`)
- KITTI (`KITTI/process_kitti.py`)
- DDAD (`DDAD/process_ddad.py`)
- DrivingStereo (`DrivingStereo/process_driving_stereo.py`)

All parsers produce aligned RGB + depth + validity-mask samples and rely on `Models/data_parsing/common/lidar_depth_fill.py`, which performs dilation, morphological closing, and median filtering on sparse depth maps.

```bash
# Argoverse
python3 Models/data_parsing/Argoverse/process_argoverse.py \
  --root /path/to/argoverse_raw/ \
  --save /path/to/processed/ARGOVERSE

# KITTI
python3 Models/data_parsing/KITTI/process_kitti.py \
  --root /path/to/kitti_raw/ \
  --save /path/to/processed/KITTI

# DDAD
python3 Models/data_parsing/DDAD/process_ddad.py \
  --data /path/to/ddad.json \
  --save /path/to/processed/DDAD

# DrivingStereo
python3 Models/data_parsing/DrivingStereo/process_driving_stereo.py \
  --root /path/to/drivingstereo_raw/ \
  --save /path/to/processed/DRIVINGSTEREO
```

#### c. Expected dataset layout

`train_scene_3d.py` loads data via `LoadDataScene3D` from `--root` and expects:

```
<ROOT_PATH>/
├── Diverse/
│   ├── image/            # RGB images
│   └── relative-depth/   # depth labels as .npy
└── Test/                 # optional images for periodic visual test outputs
```

- Image and `.npy` label counts must match exactly (one label per image, aligned in sorted filename order).
- Train/validation split is automatic: every 20th sample goes to validation, others to training.
- Depth labels are min-max normalized in `LoadDataScene3D` before being passed to `Scene3DTrainer`.

#### d. How to run training

From repository root:

```bash
# A) Fresh Scene3D run (upstream SceneSeg pre-trained)
python3 Models/training/train_scene_3d.py \
  -m /absolute/path/to/sceneseg_checkpoint.pth \
  -r /absolute/path/to/scene3d_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/

# B) Resume / fine-tune from existing Scene3D checkpoint
python3 Models/training/train_scene_3d.py \
  -l \
  -c /absolute/path/to/scene3d_checkpoint.pth \
  -r /absolute/path/to/scene3d_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/
```

During training, the script:
- Applies depth augmentations via `Augmentations(is_train=True, data_type="DEPTH")`.
- Computes total loss as `mAE_loss + 4 * multi_scale_edge_loss`.
- Uses gradient accumulation to simulate batch updates (`optimizer.step()` every `batch_size` samples).
- Logs training loss every 250 steps and training visualizations every 1000 steps.
- Runs full validation, saves checkpoints, and exports test-image predictions every `--val_interval` steps (default: `92000`).

#### e. How to visualize training results

This training pipeline provides two main visualization channels:

1. **TensorBoard logs** (training + validation scalar losses and periodic `predictions vs. actuals` figures)
2. **Saved test predictions** written to `--test_images_save_root_path` during validation checkpoints

Run TensorBoard from the repository root:

```bash
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

### 2. Exporting to ONNX and TorchScript

To export a trained `.pth` checkpoint to FP32 `.onnx` or traced TorchScript `.pt`:

```bash
# Export to FP32 ONNX
python3 Models/exports/convert_pytorch_to_onnx.py \
  -p weights/scene3d.pth \
  -o weights/scene3d.onnx

# Export to TorchScript (.pt)
python3 Models/exports/trace_pytorch_model.py \
  -p weights/scene3d.pth \
  -o weights/scene3d.pt
```